<a href="https://colab.research.google.com/github/api-sage/diaml_projects/blob/main/project6/pafolabi_DIAML_Assignment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data, Inference, and Applied Machine Learning - Assignment 6

In [ ]:
# I import all the tools at the top so I do not have to repeat imports in later cells
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, KFold, TimeSeriesSplit, cross_validate, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, mean_squared_error, r2_score

# I fix the random seed so that my splits give the same result each time I run the notebook
RANDOM_STATE = 42
plt.rcParams["figure.dpi"] = 110

## Setup and data

I use one daily file for six cities in East Africa. The cities are Kigali, Nairobi, Kampala, Dodoma, Bujumbura and Dar es Salaam. Each row is one city on one day. I load the file from the GitHub link for this project. If that link fails, I load the copy of the file that sits next to this notebook.

The file has 9,144 rows, with 1,524 days for each city, from 29 July 2022 to 29 September 2026. The first six days for each city have no air pollution values, so 36 rows have gaps in those columns. I keep those rows for Questions 2 and 3, where I only need the weather columns, and I drop them for Question 4, where I need the pollution columns.

In [ ]:
# I load the daily file and I put the rows in city and date order
data_url = "https://raw.githubusercontent.com/api-sage/diaml_projects/refs/heads/main/project6/east_africa_weather_airquality_daily.csv"
try:
    df = pd.read_csv(data_url, parse_dates=["date"])
except Exception:
    df = pd.read_csv("east_africa_weather_airquality_daily.csv", parse_dates=["date"])
df = df.sort_values(["city", "date"]).reset_index(drop=True)

# I name the groups of columns once so later cells stay short
weather_vars = ["temp_mean", "temp_max", "temp_min", "humidity_mean", "dew_point_mean",
                "pressure_mean", "cloud_cover_mean", "wind_speed_mean", "radiation_mean",
                "precipitation_sum"]
pollutant_vars = ["co", "no2", "so2", "o3", "aod", "dust"]
predictors_16 = weather_vars + pollutant_vars
cities = ["Bujumbura", "Dar es Salaam", "Dodoma", "Kampala", "Kigali", "Nairobi"]

# I show the first rows and a short summary of the file
display(df.head())
summary = pd.DataFrame({
    "item": ["rows", "cities", "first date", "last date", "days per city", "rows with no pm2_5"],
    "value": [len(df), df["city"].nunique(), str(df["date"].min().date()), str(df["date"].max().date()),
              len(df) // df["city"].nunique(), int(df["pm2_5"].isna().sum())]
})
display(summary)

### Question 1 - Nonlinearity (20 points)

#### 1.1 Can you explain what nonlinearity is and why it might be necessary to consider nonlinear relationships between variables?

I understand nonlinearity to mean that the link between two variables cannot be drawn well with one straight line. In a linear link, each extra unit of the input adds the same amount to the output, no matter where I start. In a nonlinear link, that added amount changes with the level of the input. The link can bend, it can rise fast and then flatten out, it can stay flat and then jump after a point, or it can go up and then come down.

A plain daily example helps me. If the air gets a bit more humid on a dry day, rain may still be unlikely. Once humidity and cloud pass a high level, a small extra rise can make rain much more likely. The first step and the last step do not have the same effect, so one straight line gives a poor account of what happens.

I need to allow for this shape in real data for three reasons. First, a straight line fit to a bent link gives biased forecasts. It tends to miss at the ends and in the middle in a clear pattern, and the left over errors, the residuals, still hold the bend that the model missed. Second, I can reach the wrong finding. A curved link can look like no link at all if I only fit a line. A clear case is a link that rises and then falls in the same way on both sides. The straight line slope can come out near zero even though the input tells me a lot about the output. Third, many real processes bend by their nature. Growth slows as it nears a limit, a drug has little effect at a tiny dose and no extra effect past a high dose, and rain needs moist air to cross a level before it starts.

For these reasons I first plot the data and the residuals, and I compare a straight line model with a model that can bend. If the bent model cuts the error on new data by a clear amount, I take the bend as real and I keep it in the model.

#### 1.2 Write down the mathematical equation for a nonlinear model and provide an example of an application where it might be appropriate.

I write a general nonlinear model as

$$y_i = f(x_i; \theta) + \varepsilon_i,$$

where $y_i$ is the output for case $i$, $x_i$ holds the inputs, $\theta$ holds the fixed numbers I have to learn from data, called parameters, $f$ is a function that bends, and $\varepsilon_i$ is the error. The key point is that at least one parameter enters $f$ in a nonlinear way, for example inside an exponent or inside a fraction, so I cannot rewrite the model as a simple weighted sum of known terms.

One concrete form is the growth curve

$$y = a\left(1 - e^{-bx}\right) + \varepsilon, \qquad a > 0, \; b > 0.$$

Here $a$ is the top level that $y$ moves towards and $b$ controls how fast it gets there. The parameter $b$ sits inside the exponential term, which is what makes the model nonlinear in its parameters. I note that a curve such as a squared term, $y = \beta_0 + \beta_1 x + \beta_2 x^2 + \varepsilon$, bends as a plot but stays linear in its parameters. I treat that case as a linear model with a bent input, which is not the same thing as the model above.

A good use of this form is the link between drug dose and drug effect. At a very low dose the effect is near zero. As the dose rises the effect grows fast, and past a high dose the effect flattens near a top level because the body cannot respond more. A straight line would keep rising without end and would at some point predict an effect above the top level that the body can give, which makes no sense. The curve above starts at zero, rises, and flattens at $a$, so it matches the way the body acts and each parameter has a plain meaning I can report.

#### 1.3 Can a nonlinear model be more parsimonious than a linear model? Write down mathematical formulae for both the linear and nonlinear models to support your answer.

Yes. I can give a case where the nonlinear model needs far fewer parameters. By parsimonious I mean a model that fits well with few parameters, so that each parameter earns its place.

Suppose the true link rises fast at first and then flattens near a top level, as in Question 1.2. If I insist on a model that is linear in its parameters, I have to chase the bend with many polynomial terms:

$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3 + \beta_4 x^4 + \beta_5 x^5 + \varepsilon.$$

This model has six parameters, $\beta_0$ to $\beta_5$, and I may need even more terms if the bend is sharp or the range of $x$ is wide. The fit can still wobble near the ends of the range, and the single $\beta$ values have no plain meaning on their own.

The nonlinear model for the same shape is

$$y = a\left(1 - e^{-bx}\right) + \varepsilon,$$

which has only two parameters. The value $a$ is the top level and $b$ is the speed of approach. With two parameters I get the right shape over the whole range, a clearer account of the process, and less risk of fitting noise, since a model with fewer parameters has less room to chase chance wiggles in the sample.

So I conclude that linear in parameters does not always mean simple. When the true shape is a known bend, the nonlinear form can be the more parsimonious choice. In practice I still check both models on data the model has not seen, because a small parameter count only helps if the shape is close to the truth.

#### 1.4 Surrogate data are used for testing for nonlinearity. What characteristics are typically preserved when generating surrogates? Give the names of two surrogate techniques and describe the approaches for implementing them.

I use surrogate data to ask a fair question: could a linear process have made data that looks like mine? I make many fake series that keep the linear facts of my series but break any nonlinear facts. If a nonlinear measure on my real series stands apart from the same measure on the fakes, I have proof of nonlinearity. If it sits inside the fake range, I have no such proof.

When I make surrogates I try to keep these facts of the original series: the length and the sampling steps, the mean and the spread, the power spectrum, which is the same information as the lag correlations, and, for the better methods, the full distribution of values. The null claim I test depends on the method. With plain phase randomisation the null is a linear Gaussian process. With the amplitude adjusted methods the null is wider: a linear process seen through one fixed nonlinear rescaling of the values.

The first technique I name is phase randomisation, also called Fourier transform surrogates. I take the Fourier transform of my series, which splits it into waves with sizes and phases. I keep every wave size exactly as it is and I replace the phases with random phases, drawn in a way that keeps the series real. I then invert the transform. Because I kept the wave sizes, the surrogate has the same power spectrum and the same lag correlations as the original. The values, though, become close to a Gaussian shape, and any phase links that carry nonlinear structure are broken.

The second technique is the iterated amplitude adjusted Fourier transform, or IAAFT. I start from a shuffled copy of my series so the value distribution is exact from the start. I then repeat two steps many times. In one step I impose the original power spectrum by taking the transform, swapping in the original wave sizes, and inverting. In the next step I impose the original distribution by rank ordering: I sort the current values and replace them with the sorted original values in that rank order. After many rounds the series holds both facts well. The simpler AAFT method does these steps once without the iteration, so its spectrum match is less exact.

To use either method I make, say, 99 surrogates, I compute the same nonlinear statistic on the real series and on each surrogate, for example a local forecast error, and I compare. If the real value falls outside the surrogate range, I reject the linear null at about the 1 in 100 level.

#### 1.5 Define information, entropy, and mutual information, and also provide mathematical formulas. Describe how entropy can be used for constructing a feature for measuring regularity and give an example of an application. Explain how mutual information can be used for feature selection and why it might be better than correlation.

I define information first. The information in an event is large when the event is rare and small when the event is common. For an outcome $x$ with probability $p(x)$, I write

$$I(x) = -\log_2 p(x).$$

With log base 2 the unit is bits. An event with probability one half carries 1 bit, while an event with probability one eighth carries 3 bits.

Entropy is the average information in a random variable. For a discrete variable $X$ I write

$$H(X) = -\sum_x p(x) \log_2 p(x).$$

If one outcome is almost sure, entropy is near zero because the next value holds no surprise. If many outcomes are alike in chance, entropy is high. I also use the joint entropy $H(X, Y)$ for a pair and the conditional entropy $H(X \mid Y) = H(X, Y) - H(Y)$, which is the surprise left in $X$ once I know $Y$.

Mutual information is the drop in surprise about one variable when I learn the other. I write it in two equal ways:

$$I(X; Y) = \sum_{x, y} p(x, y) \log_2 \frac{p(x, y)}{p(x)\, p(y)} = H(X) - H(X \mid Y).$$

It is zero only when $X$ and $Y$ are independent, and it grows as the link of any shape gets stronger. The formula is symmetric, so $I(X; Y) = I(Y; X)$.

To build a regularity feature from entropy, I cut a long signal into short windows and I compute an entropy value in each window. A steady, repeating signal gives low entropy because the next value is easy to guess. A messy signal gives high entropy. Forms I can use are sample entropy, which counts how often short patterns that match stay matched when I extend them by one step, and permutation entropy, which looks at the rank order of small groups of values. As an application, I can track the entropy of the gaps between heart beats from an ECG record. A heart under strain often beats in a more fixed, alike way, so the entropy falls. A fall in this feature can flag a patient who needs a closer check, long before a plain average of the same gaps would change.

For feature selection, I score each candidate input by its mutual information with the target and I keep the high scoring inputs. I prefer this to correlation when I suspect bent links, because correlation only sees straight line links. A clear case is $Y = X^2$ with $X$ spread evenly on both sides of zero. The correlation is near zero, so a correlation screen drops $X$, yet $X$ pins down $Y$ apart from sign, and the mutual information is well above zero, so an MI screen keeps it. The same gain holds for threshold links, where the output only moves once the input crosses a level. I stay aware of two limits. MI needs an estimate of probabilities from bins or from near neighbours, so it can shift with those choices and it needs more data than correlation. MI also scores each input alone, so it can keep two inputs that carry the same news. I check for that overlap after the first screen.